<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session09-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 9, Section 5 discussion — Where does the network's advantage come from?

On the book page, a 64/64 network beat the additive (linear) model on the
GB1 fitness landscape by a wide margin (test R² about 0.79 vs. 0.50). The
page's explanation: **epistasis**, the effect of a mutation depends on the
residues at the other positions.

**Test that explanation**, the way Session 7 rescued a linear model on XOR:
give the linear model **interaction features** by hand. For a pair of
positions, one extra one-hot feature for every combination of their two
amino acids (20 × 20 = 400 features per pair).

**Predict first:**

1. If all six pairs of positions get interaction features, how close does
   the linear model come to the network?
2. Which single pair of positions, among V39, D40, G41 and V54, do you
   expect to interact most?

In [1]:
import itertools, os
import numpy as np, pandas as pd
from scipy import sparse
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

CACHE = "session09-gb1.csv.gz"
path = os.path.join("data", CACHE)
gb1 = pd.read_csv(path if os.path.exists(path) else
                  "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/" + CACHE)
AAS = "ACDEFGHIKLMNPQRSTVWY"
codes = np.array([[AAS.index(a) for a in v] for v in gb1.Variants])      # 4 residue codes per variant
y = np.log10(gb1.Fitness.values + 0.001)
idx = np.arange(len(y))
rest, te = train_test_split(idx, test_size=20000, random_state=0)       # the same split as the page
tr = rest[5000:25000]                                                    # its 20,000 training variants
POS = ["V39", "D40", "G41", "V54"]

def features(pairs):
    """One-hot of each position (80 columns), plus 400 columns for each pair of positions in `pairs`."""
    n, rows = len(codes), np.arange(len(codes))
    blocks = [sparse.csr_matrix((np.ones(n), (rows, codes[:, p])), shape=(n, 20)) for p in range(4)]
    for a, b in pairs:
        blocks.append(sparse.csr_matrix((np.ones(n), (rows, codes[:, a] * 20 + codes[:, b])), shape=(n, 400)))
    return sparse.hstack(blocks).tocsr()

def test_r2(pairs):
    X = features(pairs)
    model = Ridge(alpha=1.0).fit(X[tr], y[tr])
    return X.shape[1], r2_score(y[te], model.predict(X[te]))

print(f"{'linear model with':24s} {'features':>8s} {'test R2':>8s}")
for label, pairs in [("additive only", []), ("all 6 pairs", list(itertools.combinations(range(4), 2)))]:
    n, r = test_r2(pairs)
    print(f"{label:24s} {n:8d} {r:8.3f}")
print()
for a, b in itertools.combinations(range(4), 2):
    n, r = test_r2([(a, b)])
    print(f"additive + {POS[a]} x {POS[b]:4s}     {n:8d} {r:8.3f}")

linear model with        features  test R2
additive only                  80    0.503


all 6 pairs                  2480    0.743

additive + V39 x D40           480    0.501


additive + V39 x G41           480    0.545
additive + V39 x V54           480    0.525


additive + D40 x G41           480    0.514
additive + D40 x V54           480    0.502


additive + G41 x V54           480    0.647


**Discuss:**

1. How much of the network's advantage do the six pairwise interactions
   recover? What could the rest be?
2. One pair stands out. Why might two positions interact strongly in a
   protein? (Think about residues that touch in the structure, or that
   together shape the binding surface.)
3. With 2,480 features instead of 80, why is this still a linear model?
   And why is it easy here, but impossible for longer sequences? (How
   many pair features would a 100-residue protein need?) A hidden layer
   learns the useful interactions instead of listing them all.